In [1]:
!pip install -q pandas sentence-transformers scikit-learn

In [2]:
!pip install kagglehub

In [2]:
import pandas as pd

# Update path if needed
csv_path = "analyst_ratings_processed.csv"
df = pd.read_csv(csv_path)

df.head()

,Unnamed: 0,title,date,stock
0,0.0,Stocks That Hit 52-Week Highs On Friday,2020-06-05 10:30:00-04:00,A
1,1.0,Stocks That Hit 52-Week Highs On Wednesday,2020-06-03 10:45:00-04:00,A
2,2.0,71 Biggest Movers From Friday,2020-05-26 04:30:00-04:00,A
3,3.0,46 Stocks Moving In Friday's Mid-Day Session,2020-05-22 12:45:00-04:00,A
4,4.0,B of A Securities Maintains Neutral on Agilent...,2020-05-22 11:38:00-04:00,A


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
headlines = df["title"].head(500).tolist()

embeddings = model.encode(headlines, show_progress_bar=True)
embeddings.shape  # (500, 384)
print("Embeddings shape:", embeddings )

Batches: 100%|██████████| 16/16 [00:05<00:00,  3.10it/s]


Embeddings shape: [[-8.25378485e-03 -6.54340312e-02 -7.84404390e-03 ... -1.40148178e-01
  -6.33979514e-02  7.91866183e-02]
 [-9.98138264e-03 -7.33221024e-02  1.06068808e-04 ... -1.38226420e-01
  -4.44448367e-02  8.54934603e-02]
 [-6.77622901e-03 -2.57622730e-02  2.09714435e-02 ... -1.34206042e-01
  -1.24358192e-01  6.70003518e-02]
 ...
 [-7.81287700e-02  8.50893743e-03 -3.51791307e-02 ... -1.49582282e-01
   5.80636859e-02  4.82532568e-02]
 [ 2.65659648e-03  4.12452109e-02 -8.97349417e-02 ... -1.76649928e-01
   4.76163402e-02  6.33281171e-02]
 [-1.00262035e-02  7.63014704e-03 -8.20058733e-02 ... -1.91946283e-01
   5.25472797e-02  5.30720763e-02]]


In [4]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def semantic_search(query, top_k=10):
    query_emb = model.encode([query])
    scores = cosine_similarity(query_emb, embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(headlines[i], scores[i]) for i in top_idx]

That `score` is the cosine similarity between the query embedding and each headline embedding. In the notebook, the query is encoded with `SentenceTransformer("all-MiniLM-L6-v2")`, then `cosine_similarity(query_emb, embeddings)` computes similarity against all headline vectors. The highest scores are returned.

How it’s calculated:

- Each text becomes a 384‑dimensional vector.
- Cosine similarity measures the angle between vectors:  
  $$
  \text{cosine}(a,b) = \frac{a \cdot b}{\|a\|\|b\|}
  $$
- Values are typically in $[-1, 1]$, where higher means more semantically similar.

So that `score` is just “how close” the model thinks the headline meaning is to the query, based on vector direction.

In [ ]:
queries = [
    "Federal Reserve interest rate decision",
    "Tesla earnings miss analyst expectations",
    "cryptocurrency regulation SEC enforcement",
]

for q in queries:
    print(f"\nQuery: {q}")
    for title, score in semantic_search(q, top_k=5):
        print(f"{score:.4f} | {title}")


Query: Federal Reserve interest rate decision
0.4723 | A Peek Into The Markets: US Stock Futures Up; All Eyes On Fed Decision
0.3782 | A Peek Into The Markets: U.S. Stock Futures Rise; All Eyes On Fed Minutes
0.3463 | Agilent Sees FY15 Adj. EPS $1.67-$1.73 vs $1.69 Est., Sales $4.05B-$4.111B vs $4.07B Est.; Sees Q3 Sales $995M-$1.015B vs $1B Est.
0.3401 | Bank of America Maintains Equal-weight on Agilent Technologies, Lowers PT to $42.00
0.3401 | A Peek Into The Markets: US Stock Futures Edge Higher Ahead Of Fed Speakers

Query: Tesla earnings miss analyst expectations
0.5418 | Agilent Beats Earnings Estimates, Misses On Sales And Guidance
0.4882 | Agilent Technologies Q3 Earnings Preview
0.4861 | Agilent Reports Q2 Earnings; Shares Slip
0.4842 | Q3 Earnings Preview For Agilent Technologies
0.4600 | Agilent Technologies Misses Q2 Expectations, Shares Drop

Query: cryptocurrency regulation SEC enforcement
0.3095 | A Peek Into The Markets: US Stock Futures Up; All Eyes On Fed Decision
0

In [ ]:
import re

def keyword_search(query, top_k=10):
    terms = re.findall(r"\w+", query.lower())
    scored = []
    for title in headlines:
        title_l = title.lower()
        score = sum(1 for t in terms if t in title_l)
        scored.append((title, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

for q in queries:
    print(f"\nQuery: {q}")
    print("Keyword search:")
    for title, score in keyword_search(q, top_k=5):
        print(f"{score} | {title}")


Query: Federal Reserve interest rate decision
Keyword search:
1 | Agilent Collaborates On Study Of Performance-Enhancing Spinach Extract
1 | A Peek Into The Markets: US Stock Futures Up; All Eyes On Fed Decision
1 | UPDATE: Bank Of America Reiterates On Agilent Technologies Following Investor Presentations
1 | UPDATE: Morgan Stanley Reiterates On Agilent Technologies On Balanced Risk-Reward
1 | UPDATE: Morgan Stanley Reiterates on Agilent Technologies

Query: Tesla earnings miss analyst expectations
Keyword search:
2 | Agilent Beats Earnings Estimates, Misses On Sales And Guidance
2 | Agilent Technologies Misses Q2 Expectations, Shares Drop
1 | Earnings Scheduled For May 21, 2020
1 | Shares of several healthcare companies are trading lower as markets dip following recent strength. Markets have sold off as investors weigh recent earnings and amid concerns of renewed US-China trade frictions.
1 | Agilent Technologies shares are trading lower after the company reported Q1 earnings.

Quer

### Semantic vs Keyword Search — Comparison

**Where semantic search wins**
- Captures paraphrases and implied meaning (e.g., “rate decision” vs “interest policy meeting”).
- Finds relevant headlines even when exact terms are missing or reordered.

**Where semantic search fails**
- Can surface semantically “close” but off-topic results if the query is vague.
- Misses exact-match terms that are crucial (e.g., ticker symbols) unless they appear in similar contexts.

**Where keyword search wins**
- Precise when exact terms appear (e.g., “SEC enforcement”, specific company names).
- Transparent scoring and easier to debug.

**Where keyword search fails**
- Misses synonyms and paraphrases.
- Sensitive to word form and casing if not normalized.